# L30 — Sensitivity Analysis

**Module**: M09 | **Chapter**: 12 | **Lecture**: L30

## Learning Objectives
By the end of this notebook you will be able to:
1. Construct a one-at-a-time (OAT) sensitivity tornado diagram.
2. Apply Morris screening to rank factor importance with minimal simulation budget.
3. Compute Sobol indices to apportion output variance to input factors.
4. Interpret sensitivity results to prioritise data collection and model simplification.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

System: the tandem clinic model from L29 with baseline parameters.
We ask: which input parameters most strongly affect mean sojourn time W?
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
from scipy import stats

In [ ]:
def run_tandem_params(lam, mu_reg, mu_nurse, n_clerks, n_nurses,
                      n_patients=500, warmup=100, seed=0):
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    clerks = simpy.Resource(env, capacity=n_clerks)
    nurses = simpy.Resource(env, capacity=n_nurses)
    sojourns = []

    def patient():
        t0 = env.now
        with clerks.request() as req:
            yield req
            yield env.timeout(rng.exponential(1.0 / mu_reg))
        with nurses.request() as req:
            yield req
            yield env.timeout(rng.exponential(1.0 / mu_nurse))
        sojourns.append(env.now - t0)

    def arrivals():
        for _ in range(n_patients):
            env.process(patient())
            yield env.timeout(rng.exponential(1.0 / lam))

    env.process(arrivals())
    env.run()
    return np.mean(sojourns[warmup:])


# Baseline parameters
BASE = dict(lam=5/60, mu_reg=20/60, mu_nurse=7.5/60, n_clerks=1, n_nurses=1)

W_base = np.mean([run_tandem_params(**BASE, seed=s) for s in range(20)])
print(f"Baseline W = {W_base:.3f} min")

## 1. One-at-a-Time (OAT) Sensitivity

Vary each continuous parameter ±20% from baseline while holding others fixed.
The swing (high − low) indicates importance.

In [ ]:
N_REPS_SENS = 20
DELTA = 0.20   # ±20%

# Continuous parameters to vary
cont_params = ['lam', 'mu_reg', 'mu_nurse']

oat_results = []
for param in cont_params:
    for sign, label in [(-DELTA, 'low'), (+DELTA, 'high')]:
        params = dict(BASE)
        params[param] *= (1 + sign)
        w = np.mean([run_tandem_params(**params, seed=s) for s in range(N_REPS_SENS)])
        oat_results.append({'param': param, 'level': label, 'W': w, 'delta_W': w - W_base})

oat_df = pd.DataFrame(oat_results)
print(oat_df.to_string(index=False))

In [ ]:
# Tornado diagram
fig, ax = plt.subplots(figsize=(9, 4))

# Compute swing = W_high - W_low for each parameter
swings = []
for param in cont_params:
    w_low  = oat_df[(oat_df['param']==param) & (oat_df['level']=='low')]['W'].values[0]
    w_high = oat_df[(oat_df['param']==param) & (oat_df['level']=='high')]['W'].values[0]
    swings.append({'param': param, 'w_low': w_low, 'w_high': w_high,
                   'swing': abs(w_high - w_low)})

swings_df = pd.DataFrame(swings).sort_values('swing', ascending=True)

for i, row in enumerate(swings_df.itertuples()):
    lo = min(row.w_low, row.w_high)
    hi = max(row.w_low, row.w_high)
    ax.barh(i, hi - lo, left=lo, color='steelblue', alpha=0.7)
    ax.barh(i, lo - W_base, left=W_base, color='tomato', alpha=0.7)

ax.set_yticks(range(len(swings_df)))
ax.set_yticklabels(swings_df['param'].tolist())
ax.axvline(W_base, color='black', lw=1.5, linestyle='--', label=f'Baseline W={W_base:.2f}')
ax.set_xlabel('W (min)')
ax.set_title(f'Tornado diagram — OAT sensitivity (±{int(DELTA*100)}%)')
ax.legend()
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Morris Screening

Morris screening estimates the **elementary effect** for each factor:
$$EE_i = \frac{Y(\mathbf{x} + \Delta e_i) - Y(\mathbf{x})}{\Delta}$$

where $\mathbf{x}$ is a random base point, $e_i$ is a unit vector, and $\Delta=0.5$.
Running $r$ trajectories gives $r$ elementary effects per factor.
We rank factors by $\mu^* = $ mean of |EE|.

In [ ]:
# Parameters and their [low, high] ranges
params_morris = {
    'lam':      [3/60, 7/60],
    'mu_reg':   [15/60, 25/60],
    'mu_nurse': [5/60, 10/60],
}

k = len(params_morris)
r = 10   # trajectories
DELTA_M = 0.5

rng_morris = np.random.default_rng(42)
ees = {name: [] for name in params_morris}

for traj in range(r):
    # Random starting point in [0,1]^k
    x0 = rng_morris.uniform(0, 1 - DELTA_M, size=k)
    perm = rng_morris.permutation(k)

    x_cur = x0.copy()
    names = list(params_morris.keys())
    lows  = np.array([params_morris[n][0] for n in names])
    highs = np.array([params_morris[n][1] for n in names])

    def to_real(x):
        return lows + x * (highs - lows)

    p_cur = to_real(x_cur)
    w_cur = run_tandem_params(lam=p_cur[0], mu_reg=p_cur[1], mu_nurse=p_cur[2],
                              n_clerks=1, n_nurses=1, n_patients=300, warmup=50, seed=traj)

    for j in perm:
        x_new = x_cur.copy()
        x_new[j] += DELTA_M
        p_new = to_real(x_new)
        w_new = run_tandem_params(lam=p_new[0], mu_reg=p_new[1], mu_nurse=p_new[2],
                                  n_clerks=1, n_nurses=1, n_patients=300, warmup=50,
                                  seed=traj * 100 + j)
        ee = (w_new - w_cur) / DELTA_M
        ees[names[j]].append(ee)
        x_cur = x_new
        w_cur = w_new

# Compute μ* and σ
morris_df = pd.DataFrame({
    'parameter': list(ees.keys()),
    'mu_star':   [np.mean(np.abs(v)) for v in ees.values()],
    'sigma':     [np.std(v, ddof=1) for v in ees.values()],
})
print(morris_df.sort_values('mu_star', ascending=False).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for _, row in morris_df.iterrows():
    ax.scatter(row['mu_star'], row['sigma'], s=100, zorder=3)
    ax.annotate(row['parameter'], (row['mu_star'], row['sigma']),
                textcoords='offset points', xytext=(5, 3), fontsize=10)
ax.set_xlabel('μ* (mean |EE|) — importance')
ax.set_ylabel('σ (std EE) — nonlinearity / interaction')
ax.set_title('Morris screening plot')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("High μ*, low σ  → important linear factor (good candidate for factor fixing)")
print("High μ*, high σ → important nonlinear or interacting factor")
print("Low  μ*         → unimportant factor (can be fixed at nominal value)")

## 3. Input Uncertainty Propagation

If the input parameters themselves are uncertain (estimated from data),
we propagate that uncertainty to the output W.

In [ ]:
# Suppose each continuous parameter has ±10% coefficient of variation
# We sample from truncated Normal distributions around the baseline

N_UNCERTAINTY = 100
rng_unc = np.random.default_rng(99)
cv = 0.10

W_propagated = []
for _ in range(N_UNCERTAINTY):
    lam_s    = max(0.001, rng_unc.normal(BASE['lam'],    BASE['lam']    * cv))
    mu_reg_s = max(0.001, rng_unc.normal(BASE['mu_reg'], BASE['mu_reg'] * cv))
    mu_nrs_s = max(0.001, rng_unc.normal(BASE['mu_nurse'], BASE['mu_nurse'] * cv))
    w = run_tandem_params(lam=lam_s, mu_reg=mu_reg_s, mu_nurse=mu_nrs_s,
                          n_clerks=1, n_nurses=1, seed=_)
    W_propagated.append(w)

W_prop = np.array(W_propagated)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(W_prop, bins=20, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(W_base, color='black', lw=2, linestyle='--', label=f'Nominal W={W_base:.2f}')
ax.axvline(np.percentile(W_prop, 5),  color='tomato', lw=1.5, linestyle=':', label='5th/95th pctile')
ax.axvline(np.percentile(W_prop, 95), color='tomato', lw=1.5, linestyle=':')
ax.set_xlabel('W (min)')
ax.set_ylabel('Density')
ax.set_title(f'Output uncertainty from ±{int(cv*100)}% parameter uncertainty  (n={N_UNCERTAINTY})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"W: mean={W_prop.mean():.2f}  std={W_prop.std():.2f}  90% range=[{np.percentile(W_prop,5):.2f}, {np.percentile(W_prop,95):.2f}]")

---
## Try It Yourself

1. **Spider plot**: For each parameter, compute W at 5 levels (−40%, −20%, 0%, +20%, +40%). Plot W vs. percentage change for all parameters on a single axis. Which line is steepest? Does the relationship appear linear?

2. **Rank correlation sensitivity**: Run 200 random samples across all parameter ranges simultaneously (Latin Hypercube Sampling). Compute the Spearman rank correlation between each input and W. Compare the ranking to the Morris μ* result.

3. **Fixed-vs-free decision**: Suppose you can afford to collect 200 additional observations for exactly one parameter. Based on the Morris μ* ranking, which parameter's uncertainty most reduces output variance? Verify by re-running the uncertainty propagation with that parameter fixed at its baseline value.